# Esercizio 1 - Circuito a 1 qubit con H e misura

- `measure_all()` dichiara i cbit necessari da solo, quindi si potrebbe mettere anche `qc=QuantumCircuit(1)`
- Perchè non basta `measure`? Perchè bisogna usare un silmulatore? 
    * il QuantumCircuit è una descrizione, non un'esecuzione. Una lista ordinata di istruzioni
    * è il backend che parte da uno stato e gli applica i vari gate seguendo le istruzioni del codice
- `BasicAer` comando che è stato tolto in Qiskit v1.0 -> sostituito con `BasicProvider`
    - "DeprecationWarning: BasicAer is deprecated since Qiskit 0.46 and will be removed in Qiskit 1.0. The BasicAer (qiskit.providers.basicaer) module has been superseded by  qiskit.providers.basic_provider, and all its classes have been renamed to follow a new naming convention. More information and migration guidelines can be found in the 0.46 API docs for BasicAer.
    - Fonte: https://quantum.cloud.ibm.com/docs/guides/qiskit-1.0-features


- RISULTATO: Esempio concettuale di output: `{'00': 495, '11': 505}`
    - significa che su 1000 shot totali, lo stato "00" è stato misurato 495 volte e lo stato "11" 505 volte
    - questo risultato è tipico di un circuito che crea uno stato di Bell (sovrapposizione entangled tra i due esiti "00" e "11")

In [30]:

from qiskit import QuantumCircuit, transpile
from qiskit.providers.basic_provider import BasicProvider
from qiskit.quantum_info import Statevector, Operator

qc=QuantumCircuit(1,1)
qc.h(0)
qc.measure(0,0)

backend = BasicProvider().get_backend("basic_simulator")
tqc = transpile(qc, backend)
job = backend.run(tqc, shots=1000)
counts = job.result().get_counts()
print(counts)



{'1': 527, '0': 473}


- Dalla v1.0 i backend `statevector_simulator` e `unitary_simulator` usati nel libro sono stati rimossi perché duplicavano funzionalità già presenti in `qiskit.quantum_info`: si usano quindi direttamente `Statevector(qc)` e `Operator(qc)`.
    * Per vettore di stato e matrice unitaria non serve più il passaggio per il backend perché sono calcoli matematici deterministici, non esecuzioni
- IMPORTANTE: Statevector e Operator NON accetta circuiti che contengono misure
    * vettore di stato e matrice unitaria descrivono un'evoluzione unitaria, mentre la misura fa collassare lo stato
- RISULTATO: 
    1. Esempio concettuale di output: $[0.7071+0.j, 0.+0.j, 0.+0.j, 0.7071+0.j]$
        - sono le quattro ampiezze di probabilità per gli stati $|00⟩, |01⟩, |10⟩, |11⟩$
        - il valore 0.7071 corrisponde circa a 1 diviso radice quadrata di 2, tipico di una sovrapposizione bilanciata → |00⟩ e |11⟩ hanno la stessa ampiezza 1/√2
        - Il punto concettualmente decisivo però è che le ampiezze di |01⟩ e |10⟩ sono **zero →** Questo stato non è solo sovrapposto: è **entangled**
            - È lo stato di Bell $Φ⁺$, scritto $\frac{|00⟩ + |11⟩}{\sqrt{2}}$
    2. L'output è una matrice quadrata di numeri complessi (una riga e una colonna per ogni stato di base)

In [28]:
from qiskit.quantum_info import Statevector, Operator

qc=QuantumCircuit(1,1)
qc.h(0)

statevector = Statevector(qc)
print(statevector)

unitary = Operator(qc)
print(unitary)

Statevector([0.70710678+0.j, 0.70710678+0.j],
            dims=(2,))
Operator([[ 0.70710678+0.j,  0.70710678+0.j],
          [ 0.70710678+0.j, -0.70710678+0.j]],
         input_dims=(2,), output_dims=(2,))
